# Sketch → photo end-to-end rehabilitation (v3, resumable)

This notebook trains **only pencil sketch → colour photo**. It keeps the accepted
photo → pencil model untouched and never downloads the persistent COCO dataset again.

Why this replaces the v2 refiner: the v2 experiment froze a pink, blurry coarse U-Net
and permitted only a small residual correction. Its step-2000 visual gate showed that
the correction could not repair identity or colour. V3 instead fine-tunes the complete
generator with paired reconstruction first, then introduces a low-weight PatchGAN.

There is no guarantee that a monochrome sketch can recover the exact real-world colour
or identity. Validation therefore tests FS2K artist portraits and COCO general scenes
separately and preserves the best checkpoint without overwriting older work.

In [ ]:
# Install only the small packages missing from a normal Colab GPU runtime.
# Do not replace Colab's torch/torchvision wheels.
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'lpips==0.1.4', 'gdown==5.2.0',
])
print('Dependencies ready.')


In [ ]:
from pathlib import Path
from collections import defaultdict
from copy import deepcopy
import hashlib, json, math, os, random, shutil, time, zipfile

import cv2
import gdown
import lpips
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

assert torch.cuda.is_available(), 'Choose Runtime → Change runtime type → GPU, then rerun.'
DEVICE = torch.device('cuda')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False

# Stage A repairs structure/colour at 256 px without a GAN. Stage B adds restrained
# adversarial texture and moves to 512 px. Every checkpoint is resumable from Drive.
TOTAL_UPDATES = 8_000
GAN_START = 1_500
START_512_AT = 4_000
SAVE_EVERY = 250
VALIDATE_EVERY = 500
FS2K_SAMPLE_PROBABILITY = 0.70
VALIDATION_PER_DOMAIN = 20
EARLY_STOPPING_VALIDATIONS = 8

from google.colab import drive
drive.mount('/content/drive')

SAVE_ROOT = Path('/content/drive/MyDrive/Sketch2Photo_training')
COCO_ROOT = SAVE_ROOT / 'datasets' / 'coco2017_general_20k'
FS2K_STORAGE = SAVE_ROOT / 'datasets' / 'FS2K_persistent'
RUN_ROOT = SAVE_ROOT / 'focused_sketch_to_photo_v3_end_to_end'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
FS2K_STORAGE.mkdir(parents=True, exist_ok=True)

STARTING_CANDIDATES = [
    SAVE_ROOT / 'general_20k_bidirectional_v1/general_rescue/unet_multiscale_patchgan/sketch_to_photo/best.pt',
    SAVE_ROOT / 'general_20k_sketch_photo_v1/general_rescue/unet_multiscale_patchgan/sketch_to_photo/best.pt',
    SAVE_ROOT / 'face_specific_rescue_v1/face_rescue/unet/sketch_to_photo/best.pt',
]
STARTING_CHECKPOINT = next((path for path in STARTING_CANDIDATES if path.is_file()), None)
assert STARTING_CHECKPOINT is not None, 'No completed sketch→photo U-Net checkpoint was found in Drive.'

print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')
print('Starting checkpoint:', STARTING_CHECKPOINT)
print('New v3 run folder:', RUN_ROOT)
print('COCO will NOT be downloaded by this notebook.')
print('The photo→pencil model will NOT be changed.')

## 1. Discover and audit the existing paired data

This cell fails instead of silently downloading COCO when the persistent 20K folder is
missing. FS2K is stored in Drive so a later Colab runtime can reuse it.


In [ ]:
ALLOWED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, default=str))
    os.replace(temporary, path)

def atomic_torch(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    torch.save(value, temporary)
    os.replace(temporary, path)

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def safe_extract(zip_path, destination):
    destination = Path(destination); destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for entry in archive.infolist():
            resolved = (destination / entry.filename).resolve()
            if not resolved.is_relative_to(root) or ((entry.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError('Unsafe ZIP entry: ' + entry.filename)
        archive.extractall(destination)

def find_fs2k_root():
    candidates = list(FS2K_STORAGE.rglob('anno_train.json'))
    if not candidates:
        archive = FS2K_STORAGE / 'FS2K.zip'
        if not archive.is_file():
            print('FS2K is absent from Drive; downloading it once to persistent storage...')
            gdown.download(
                id='1saIMhQ3dc5_ftkfGmBPbCluRn_zy7QQp',
                output=str(archive), quiet=False,
            )
        if not zipfile.is_zipfile(archive):
            raise ValueError('The downloaded FS2K file is not a valid ZIP.')
        safe_extract(archive, FS2K_STORAGE / 'extracted')
        candidates = list(FS2K_STORAGE.rglob('anno_train.json'))
    assert len(candidates) == 1, f'Expected one FS2K root, found {len(candidates)}.'
    root = candidates[0].parent
    assert (root / 'anno_test.json').is_file(), 'FS2K test annotations are missing.'
    return root

def normalize_stem(value):
    return Path(str(value).replace('\\', '/')).with_suffix('').as_posix().casefold()

def image_index(root, folder):
    base = root / folder
    index = defaultdict(list)
    for path in base.rglob('*'):
        if path.is_file() and path.suffix.lower() in ALLOWED_EXTENSIONS:
            index[normalize_stem(path.relative_to(base))].append(path)
    assert index, f'No images found in {base}'
    return index

def build_fs2k_records(root):
    train_rows = json.loads((root / 'anno_train.json').read_text())
    test_rows = json.loads((root / 'anno_test.json').read_text())
    grouped = defaultdict(list)
    for row in train_rows:
        grouped[str(row['style'])].append(row)
    tagged = []
    for style, rows in sorted(grouped.items()):
        rows = sorted(rows, key=lambda row: row['image_name'])
        random.Random(SEED + int(hashlib.sha256(style.encode()).hexdigest()[:8], 16)).shuffle(rows)
        cut = max(1, round(0.2 * len(rows)))
        tagged += [(row, 'val') for row in rows[:cut]]
        tagged += [(row, 'train') for row in rows[cut:]]
    tagged += [(row, 'test') for row in test_rows]

    photos, sketches = image_index(root, 'photo'), image_index(root, 'sketch')
    records, excluded = [], []
    for row, split in tagged:
        photo_stem = Path(str(row['image_name']).replace('\\', '/')).with_suffix('').as_posix()
        parts = photo_stem.split('/')
        parts[0] = parts[0].replace('photo', 'sketch', 1)
        sketch_path = Path('/'.join(parts))
        name = sketch_path.name
        if name.startswith('image'):
            name = 'sketch' + name[len('image'):]
        sketch_stem = (sketch_path.parent / name).as_posix()
        photo_matches = photos.get(normalize_stem(photo_stem), [])
        sketch_matches = sketches.get(normalize_stem(sketch_stem), [])
        if len(photo_matches) != 1 or len(sketch_matches) != 1:
            excluded.append(str(row['image_name']))
            continue
        records.append({
            'id': 'fs2k_' + photo_stem,
            'split': split,
            'domain': 'fs2k_artist',
            'photo': str(photo_matches[0]),
            'sketch': str(sketch_matches[0]),
        })
    assert len(records) >= int(0.98 * len(tagged)), (
        f'Only {len(records)}/{len(tagged)} FS2K pairs were resolved. Check the archive.'
    )
    return records, excluded

def build_coco_records():
    manifest_path = COCO_ROOT / 'manifest.json'
    photo_root = COCO_ROOT / 'photos'
    assert manifest_path.is_file(), (
        f'Missing {manifest_path}. Copy/share the completed dataset folder; this notebook will not redownload COCO.'
    )
    manifest = json.loads(manifest_path.read_text())
    records = []
    for row in manifest.get('records', []):
        filename = row.get('file_name') or Path(row.get('photo', '')).name
        photo = photo_root / filename
        if photo.is_file():
            records.append({
                'id': str(row.get('id', 'coco_' + photo.stem)),
                'split': row['split'],
                'domain': 'coco_synthetic',
                'photo': str(photo),
                'sketch': '',
            })
    train_count = sum(row['split'] == 'train' for row in records)
    assert train_count >= 20_000, (
        f'Only {train_count}/20000 persistent COCO training photos were found. '
        'Do not train on an incomplete folder.'
    )
    return records

FS2K_ROOT = find_fs2k_root()
FS2K_RECORDS, FS2K_EXCLUDED = build_fs2k_records(FS2K_ROOT)
COCO_RECORDS = build_coco_records()

FS2K_SPLITS = {split: [r for r in FS2K_RECORDS if r['split'] == split] for split in ('train','val','test')}
COCO_SPLITS = {split: [r for r in COCO_RECORDS if r['split'] == split] for split in ('train','val','test')}
for split in ('train', 'val', 'test'):
    print(split, '| FS2K:', len(FS2K_SPLITS[split]), '| COCO:', len(COCO_SPLITS[split]))

DATA_FINGERPRINT = hashlib.sha256(json.dumps({
    'fs2k': [(r['id'], r['split']) for r in FS2K_RECORDS],
    'coco': [(r['id'], r['split']) for r in COCO_RECORDS],
    'seed': SEED,
}, sort_keys=True).encode()).hexdigest()
atomic_json(RUN_ROOT / 'data_audit.json', {
    'fs2k_root': str(FS2K_ROOT),
    'fs2k_usable': len(FS2K_RECORDS),
    'fs2k_excluded': FS2K_EXCLUDED,
    'coco_root': str(COCO_ROOT),
    'coco_usable': len(COCO_RECORDS),
    'fingerprint': DATA_FINGERPRINT,
})
print('Data fingerprint:', DATA_FINGERPRINT)


## 2. Paired preprocessing

Real FS2K sketches remain real artist inputs. COCO photos receive a deterministic
graphite conversion only to create aligned training conditions; the original photo is
always the target. Geometry-changing augmentation is applied identically to both sides.


In [ ]:
def fit_canvas(image, size):
    image = ImageOps.exif_transpose(image).convert('RGB')
    scale = min(size / image.width, size / image.height)
    width, height = max(1, round(image.width * scale)), max(1, round(image.height * scale))
    resized = image.resize((width, height), Image.Resampling.LANCZOS)
    canvas = Image.new('RGB', (size, size), 'white')
    left, top = (size - width) // 2, (size - height) // 2
    canvas.paste(resized, (left, top))
    return canvas, (left, top, left + width, top + height)

def make_clean_line_art(photo):
    rgb = np.asarray(photo.convert('RGB'))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    smooth = cv2.bilateralFilter(gray, 7, 32, 7)
    inverted = 255 - smooth
    blurred = cv2.GaussianBlur(inverted, (0, 0), 5.0)
    dodge = cv2.divide(smooth, np.maximum(255 - blurred, 1).astype(np.uint8), scale=256)
    fine = cv2.GaussianBlur(smooth, (0, 0), .65).astype(np.float32)
    medium = cv2.GaussianBlur(smooth, (0, 0), 1.65).astype(np.float32)
    edges = np.maximum(medium - fine, 0) / 255.0
    shadow = np.power((255 - smooth.astype(np.float32)) / 255.0, 1.45)
    density = .74 * ((255 - dodge.astype(np.float32)) / 255.0) + .22 * shadow + 2.1 * edges
    drawing = np.clip(254 - 255 * (1 - np.exp(-2.05 * density)), 8, 255).astype(np.uint8)
    return Image.fromarray(drawing).convert('RGB')

def augment_sketch(image, rng):
    sketch = ImageOps.autocontrast(image.convert('L'), cutoff=.2)
    sketch = ImageEnhance.Contrast(sketch).enhance(rng.uniform(.88, 1.15))
    sketch = ImageEnhance.Brightness(sketch).enhance(rng.uniform(.97, 1.03))
    if rng.random() < .15:
        sketch = sketch.filter(ImageFilter.MinFilter(3))
    elif rng.random() < .15:
        sketch = sketch.filter(ImageFilter.GaussianBlur(rng.uniform(.15, .45)))
    array = np.asarray(sketch, dtype=np.float32)
    noise = np.random.default_rng(rng.randrange(2**32)).normal(0, rng.uniform(0, 1.0), array.shape)
    return Image.fromarray(np.clip(array + noise, 0, 255).astype(np.uint8)).convert('RGB')

def tensor_image(image):
    array = np.asarray(image, dtype=np.float32).copy() / 127.5 - 1.0
    return torch.from_numpy(array.transpose(2, 0, 1))

def load_pair(record, resolution):
    with Image.open(record['photo']) as image:
        photo, box = fit_canvas(image, resolution)
    if record['domain'] == 'fs2k_artist':
        with Image.open(record['sketch']) as image:
            sketch, sketch_box = fit_canvas(image, resolution)
        assert sketch_box == box, 'FS2K pair geometry does not match.'
        sketch = sketch.convert('L').convert('RGB')
    else:
        sketch = make_clean_line_art(photo)
    return photo, sketch, box

class FocusedDataset(Dataset):
    def __init__(self, total_indices, resolution, training=True):
        self.total_indices = total_indices
        self.resolution = resolution
        self.training = training

    def __len__(self):
        return len(self.total_indices)

    def __getitem__(self, position):
        global_index = self.total_indices[position]
        rng = random.Random(SEED + 1_000_003 * global_index)
        use_fs2k = rng.random() < FS2K_SAMPLE_PROBABILITY
        pool = FS2K_SPLITS['train'] if use_fs2k else COCO_SPLITS['train']
        record = pool[rng.randrange(len(pool))]
        photo, sketch, _ = load_pair(record, self.resolution)
        if self.training and rng.random() < .5:
            photo, sketch = ImageOps.mirror(photo), ImageOps.mirror(sketch)
        if self.training and rng.random() < .35:
            sketch = augment_sketch(sketch, rng)
        return {
            'source': tensor_image(sketch),
            'target': tensor_image(photo),
            'domain': record['domain'],
            'id': record['id'],
        }

VALIDATION_RECORDS = (
    FS2K_SPLITS['val'][:VALIDATION_PER_DOMAIN]
    + COCO_SPLITS['val'][:VALIDATION_PER_DOMAIN]
)
assert len(VALIDATION_RECORDS) == 2 * VALIDATION_PER_DOMAIN

fig, axes = plt.subplots(4, 2, figsize=(9, 15))
preview_records = FS2K_SPLITS['train'][:2] + COCO_SPLITS['train'][:2]
for row, record in enumerate(preview_records):
    photo, sketch, box = load_pair(record, 512)
    for ax, image, title in zip(axes[row], [sketch.crop(box), photo.crop(box)], ['Sketch input', 'Photo target']):
        ax.imshow(image); ax.set_title(record['domain'] + ' — ' + title); ax.axis('off')
plt.tight_layout()
plt.savefig(RUN_ROOT / 'pair_audit.png', dpi=130)
plt.show()
print('STOP if any row above is mismatched. Otherwise continue.')


## 3. End-to-end generator and two PatchGAN critics

The completed sketch→photo U-Net is used only as an initialization. Unlike the rejected
v2 experiment, **all generator layers are trainable**, so the network can remove the
magenta cast and rebuild blurred structure. An exponential-moving-average copy is used
for validation. The two discriminators operate at full and half scale and begin only
after the reconstruction-only warm-up.

In [ ]:
def unet_block(input_channels, output_channels):
    return nn.Sequential(
        nn.Conv2d(input_channels, output_channels, 3, padding=1),
        nn.GroupNorm(8, output_channels), nn.SiLU(),
        nn.Conv2d(output_channels, output_channels, 3, padding=1),
        nn.GroupNorm(8, output_channels), nn.SiLU(),
    )

class PencilUNet(nn.Module):
    def __init__(self, width=64, output_mode='direct'):
        super().__init__()
        self.output_mode = output_mode
        self.enc = nn.ModuleList([
            unet_block(3, width), unet_block(width, width*2),
            unet_block(width*2, width*4), unet_block(width*4, width*8),
        ])
        self.mid = unet_block(width*8, width*8)
        self.dec = nn.ModuleList([
            unet_block(width*16, width*4), unet_block(width*8, width*2),
            unet_block(width*4, width), unet_block(width*2, width),
        ])
        self.out = nn.Conv2d(width, 3, 1)

    def forward(self, x):
        source, skips = x, []
        for layer in self.enc:
            x = layer(x); skips.append(x); x = F.avg_pool2d(x, 2)
        x = self.mid(x)
        for layer, skip in zip(self.dec, reversed(skips)):
            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
            x = layer(torch.cat([x, skip], 1))
        prediction = self.out(x)
        return (source + prediction).clamp(-1, 1) if self.output_mode == 'residual' else torch.tanh(prediction)

class FeaturePatchDiscriminator(nn.Module):
    def __init__(self, width=48):
        super().__init__()
        channels = [6, width, width*2, width*4, width*8]
        self.blocks = nn.ModuleList()
        for index in range(4):
            stride = 2 if index < 3 else 1
            self.blocks.append(nn.Sequential(
                nn.utils.spectral_norm(nn.Conv2d(channels[index], channels[index+1], 4, stride, 1)),
                nn.LeakyReLU(.2, inplace=True),
            ))
        self.out = nn.utils.spectral_norm(nn.Conv2d(channels[-1], 1, 4, 1, 1))

    def forward(self, paired):
        features = []
        for block in self.blocks:
            paired = block(paired); features.append(paired)
        return self.out(paired), features

starting_state = torch.load(STARTING_CHECKPOINT, map_location='cpu', weights_only=True)
assert starting_state.get('direction') == 'sketch_to_photo'
generator_width = int(starting_state.get('width', 64))
generator_mode = starting_state.get('output_mode', 'direct')

generator = PencilUNet(generator_width, generator_mode).to(DEVICE)
generator.load_state_dict(starting_state['generator'], strict=True)
generator.train().requires_grad_(True)

# Immutable baseline for honest before/after comparisons.
baseline_generator = deepcopy(generator).eval().requires_grad_(False)
ema_generator = deepcopy(generator).eval().requires_grad_(False)
discriminator_full = FeaturePatchDiscriminator().to(DEVICE)
discriminator_half = FeaturePatchDiscriminator().to(DEVICE)

print('Loaded starting step:', starting_state.get('step'))
print('Generator output mode:', generator_mode)
print('Trainable generator parameters:', sum(p.numel() for p in generator.parameters()))
print('Important: the whole generator is trainable in v3.')

## 4. Portrait-aware validation and colour-safe losses

The loss strongly prioritizes paired fidelity, grayscale structure, edges, and RGB
statistics. This directly targets the pink cast and blurred facial geometry. PatchGAN
texture is delayed and deliberately weak. Validation reports colour error separately,
weights FS2K portraits slightly more heavily, and rejects checkpoints that damage either
domain relative to the starting U-Net.

In [ ]:
PERCEPTUAL = lpips.LPIPS(net='alex', version='0.1').to(DEVICE).eval().requires_grad_(False)

def spatial_gradients(x):
    return x[:, :, :, 1:] - x[:, :, :, :-1], x[:, :, 1:, :] - x[:, :, :-1, :]

def colour_statistics(x):
    return x.mean((2,3)), x.std((2,3), unbiased=False)

def rgb_luma(x):
    return .299*x[:,0:1] + .587*x[:,1:2] + .114*x[:,2:3]

def charbonnier(prediction, target, epsilon=1e-3):
    return torch.sqrt((prediction - target).square() + epsilon**2).mean()

def reconstruction_terms(fake, target):
    # All loss arithmetic, especially LPIPS, remains float32 to avoid T4 overflow.
    fake, target = fake.float(), target.float()
    fake_dx, fake_dy = spatial_gradients(fake)
    target_dx, target_dy = spatial_gradients(target)
    fake_mean, fake_std = colour_statistics(fake)
    target_mean, target_std = colour_statistics(target)
    fake_luma, target_luma = rgb_luma(fake), rgb_luma(target)
    with torch.autocast('cuda', enabled=False):
        perceptual = PERCEPTUAL(fake, target).mean()
    terms = {
        'pixel': charbonnier(fake, target),
        'perceptual': perceptual,
        'luma': charbonnier(fake_luma, target_luma),
        'edge': charbonnier(fake_dx, target_dx) + charbonnier(fake_dy, target_dy),
        'colour': F.l1_loss(fake_mean, target_mean) + F.l1_loss(fake_std, target_std),
    }
    terms['total'] = (
        7.0*terms['pixel'] + 1.0*terms['perceptual'] + 2.0*terms['luma']
        + 1.5*terms['edge'] + 2.5*terms['colour']
    )
    return terms

@torch.no_grad()
def predict_tensor(sketch_tensor, active_generator):
    baseline = baseline_generator(sketch_tensor)
    prediction = active_generator(sketch_tensor)
    return baseline, prediction

@torch.no_grad()
def validate(active_generator):
    active_generator.eval(); rows = []
    for record in tqdm(VALIDATION_RECORDS, desc='Validation', leave=False):
        photo, sketch, _ = load_pair(record, 512)
        source = tensor_image(sketch)[None].to(DEVICE)
        target = tensor_image(photo)[None].to(DEVICE)
        with torch.autocast('cuda', dtype=torch.float16):
            _, prediction = predict_tensor(source, active_generator)
        prediction_f, target_f = prediction.float(), target.float()
        with torch.autocast('cuda', enabled=False):
            lpips_value = float(PERCEPTUAL(prediction_f, target_f).mean())
        mae = float(F.l1_loss(prediction_f, target_f) / 2)
        pred_dx, pred_dy = spatial_gradients(prediction_f)
        target_dx, target_dy = spatial_gradients(target_f)
        edge = float((F.l1_loss(pred_dx, target_dx) + F.l1_loss(pred_dy, target_dy)) / 4)
        pred_mean, pred_std = colour_statistics(prediction_f)
        target_mean, target_std = colour_statistics(target_f)
        colour = float((F.l1_loss(pred_mean, target_mean) + F.l1_loss(pred_std, target_std)) / 4)
        rows.append({'id':record['id'], 'domain':record['domain'], 'lpips':lpips_value,
                     'mae':mae, 'edge':edge, 'colour':colour})
    summary = {}
    for domain in ('fs2k_artist', 'coco_synthetic'):
        domain_rows = [row for row in rows if row['domain'] == domain]
        summary[domain] = {
            metric: float(np.mean([row[metric] for row in domain_rows]))
            for metric in ('lpips','mae','edge','colour')
        }
        values = summary[domain]
        values['score'] = values['lpips'] + .35*values['mae'] + .10*values['edge'] + .25*values['colour']
    summary['balanced_score'] = float(.60*summary['fs2k_artist']['score'] + .40*summary['coco_synthetic']['score'])
    return summary, rows

def tensor_to_image(tensor):
    array = ((tensor.detach().float().cpu().clamp(-1,1)[0].permute(1,2,0).numpy()+1)*127.5)
    return Image.fromarray(array.round().astype('uint8'))

def render_gate(active_generator, step, filename):
    active_generator.eval()
    records = FS2K_SPLITS['val'][:3] + COCO_SPLITS['val'][:3]
    fig, axes = plt.subplots(len(records), 4, figsize=(14, 3.7*len(records)))
    for row, record in enumerate(records):
        photo, sketch, box = load_pair(record, 512)
        source = tensor_image(sketch)[None].to(DEVICE)
        with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
            baseline, prediction = predict_tensor(source, active_generator)
        images = [sketch.crop(box), tensor_to_image(baseline).crop(box),
                  tensor_to_image(prediction).crop(box), photo.crop(box)]
        titles = ['Input sketch', 'Starting U-Net', f'End-to-end step {step}', 'Real target']
        for axis, image, title in zip(axes[row], images, titles):
            axis.imshow(image); axis.set_title(title); axis.axis('off')
    fig.tight_layout(); fig.savefig(RUN_ROOT / filename, dpi=130); plt.show(); plt.close(fig)

## 5. Train or resume the v3 generator

Run this cell. A fresh v3 folder begins at step 0; rerunning after a Colab disconnect
resumes automatically. Steps 1–1499 use only stable paired losses. PatchGAN begins at
step 1500, and 512-pixel fine-tuning begins at step 4000. Inspect every 500-step gate.

**Stop early** if three consecutive gates visibly worsen faces or scenes. More updates
cannot guarantee quality, and the original checkpoint is always preserved.

In [ ]:
RECIPE = {
    'format': 'sketch2photo-end-to-end-v3', 'seed': SEED,
    'total_updates': TOTAL_UPDATES, 'gan_start': GAN_START,
    'start_512_at': START_512_AT, 'save_every': SAVE_EVERY,
    'validate_every': VALIDATE_EVERY, 'fs2k_probability': FS2K_SAMPLE_PROBABILITY,
    'starting_checkpoint': str(STARTING_CHECKPOINT),
    'starting_step': int(starting_state.get('step', -1)),
    'data_fingerprint': DATA_FINGERPRINT,
    'generator_lr': 2e-5, 'discriminator_lr': 4e-5,
    'adversarial_max': .025, 'adversarial_warmup': 2_000,
    'implementation': 'v3-end-to-end-fp32-loss-safe-amp',
}
SIGNATURE = hashlib.sha256(json.dumps(RECIPE, sort_keys=True).encode()).hexdigest()
atomic_json(RUN_ROOT / 'recipe.json', RECIPE | {'signature':SIGNATURE})

optimizer_g = torch.optim.AdamW(generator.parameters(), lr=RECIPE['generator_lr'], betas=(.5,.999), weight_decay=1e-5)
discriminator_parameters = list(discriminator_full.parameters()) + list(discriminator_half.parameters())
optimizer_d = torch.optim.AdamW(discriminator_parameters, lr=RECIPE['discriminator_lr'], betas=(.5,.999))
scheduler_g = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_g, T_max=TOTAL_UPDATES, eta_min=2e-6)
scheduler_d = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_d, T_max=max(1,TOTAL_UPDATES-GAN_START), eta_min=4e-6)
scaler = torch.amp.GradScaler('cuda', init_scale=128.0, growth_factor=2.0, backoff_factor=.5, growth_interval=2_000)

progress = {'step':0, 'best_step':0, 'best_score':float('inf'), 'bad_validations':0,
            'seconds':0.0, 'baseline':None, 'quality_accepted':False}
history = []
last_path = RUN_ROOT / 'last_training_state.pt'

if last_path.is_file():
    state = torch.load(last_path, map_location='cpu', weights_only=False)
    assert state['signature'] == SIGNATURE, 'V3 configuration changed. Restore it or choose a new RUN_ROOT.'
    generator.load_state_dict(state['generator']); ema_generator.load_state_dict(state['ema_generator'])
    discriminator_full.load_state_dict(state['discriminator_full']); discriminator_half.load_state_dict(state['discriminator_half'])
    optimizer_g.load_state_dict(state['optimizer_g']); optimizer_d.load_state_dict(state['optimizer_d'])
    scheduler_g.load_state_dict(state['scheduler_g']); scheduler_d.load_state_dict(state['scheduler_d'])
    scaler.load_state_dict(state['scaler']); progress, history = state['progress'], state['history']
    torch.set_rng_state(state['torch_rng']); torch.cuda.set_rng_state_all(state['cuda_rng'])
    random.setstate(state['python_rng']); np.random.set_state(state['numpy_rng'])
    print('Resumed v3 safely at update', progress['step'])

def cpu_state_dict(model):
    return {key:value.detach().cpu() for key,value in model.state_dict().items()}

def save_deploy_checkpoint(path, summary, accepted):
    atomic_torch(path, {
        'format':'sketch2photo-end-to-end-v3', 'direction':'sketch_to_photo',
        'resolution':512, 'step':progress['step'], 'validation':summary,
        'quality_accepted':bool(accepted), 'width':generator_width,
        'output_mode':generator_mode, 'generator':cpu_state_dict(ema_generator),
        'starting_checkpoint':str(STARTING_CHECKPOINT),
        'data_fingerprint':DATA_FINGERPRINT, 'recipe_signature':SIGNATURE,
    })

def save_snapshot(path, summary):
    atomic_torch(path, {'format':'sketch2photo-end-to-end-snapshot-v3',
        'direction':'sketch_to_photo', 'step':progress['step'], 'validation':summary,
        'width':generator_width, 'output_mode':generator_mode,
        'generator':cpu_state_dict(ema_generator), 'recipe_signature':SIGNATURE})

started = time.perf_counter(); elapsed_before = progress['seconds']
def save_last():
    progress['seconds'] = elapsed_before + time.perf_counter() - started
    atomic_torch(last_path, {
        'signature':SIGNATURE, 'generator':generator.state_dict(),
        'ema_generator':ema_generator.state_dict(),
        'discriminator_full':discriminator_full.state_dict(),
        'discriminator_half':discriminator_half.state_dict(),
        'optimizer_g':optimizer_g.state_dict(), 'optimizer_d':optimizer_d.state_dict(),
        'scheduler_g':scheduler_g.state_dict(), 'scheduler_d':scheduler_d.state_dict(),
        'scaler':scaler.state_dict(), 'progress':dict(progress), 'history':history,
        'torch_rng':torch.get_rng_state(), 'cuda_rng':torch.cuda.get_rng_state_all(),
        'python_rng':random.getstate(), 'numpy_rng':np.random.get_state(),
    })
    atomic_json(RUN_ROOT / 'history.json', history); atomic_json(RUN_ROOT / 'progress.json', progress)

def run_validation():
    summary, rows = validate(ema_generator)
    history.append({'type':'validation', 'step':progress['step'], **summary})
    atomic_json(RUN_ROOT / f"validation_{progress['step']:07d}.json", {'summary':summary,'rows':rows})
    if progress['baseline'] is None:
        progress['baseline'] = summary
    baseline = progress['baseline']
    portrait_gain = summary['fs2k_artist']['lpips'] <= baseline['fs2k_artist']['lpips']*.97
    colour_gain = summary['fs2k_artist']['colour'] <= baseline['fs2k_artist']['colour']*.95
    domain_guard = all(
        summary[d]['lpips'] <= baseline[d]['lpips']*1.05 and summary[d]['mae'] <= baseline[d]['mae']*1.05
        for d in ('fs2k_artist','coco_synthetic')
    )
    accepted = progress['step'] >= 500 and portrait_gain and colour_gain and domain_guard
    improved = domain_guard and summary['balanced_score'] < progress['best_score'] - 1e-5
    # If the best numeric candidate missed the minimum portrait/colour thresholds,
    # preserve the first candidate that passes them even when its blended score is
    # fractionally higher. Deployment quality takes priority over one scalar.
    selected = improved or (accepted and not progress['quality_accepted'])
    if selected:
        progress['best_score'] = summary['balanced_score']; progress['best_step'] = progress['step']
        progress['quality_accepted'] = bool(accepted); progress['bad_validations'] = 0
        save_deploy_checkpoint(RUN_ROOT / 'best.pt', summary, accepted)
    else:
        progress['bad_validations'] += 1
    save_snapshot(RUN_ROOT / f"snapshot_{progress['step']:07d}.pt", summary)
    render_gate(ema_generator, progress['step'], f"gate_{progress['step']:07d}.png")
    print('Validation:', json.dumps(summary, indent=2))
    print('Selected checkpoint:', selected, '| best step:', progress['best_step'])
    print('Automatic quality gate passed:', accepted)

if not history:
    run_validation(); save_last()

bar = tqdm(total=TOTAL_UPDATES, initial=progress['step'], desc='V3 end-to-end sketch→photo')
stop_early = False; consecutive_nonfinite = 0
while progress['step'] < TOTAL_UPDATES and not stop_early:
    resolution = 256 if progress['step'] < START_512_AT else 512
    boundary = min(TOTAL_UPDATES, START_512_AT if resolution == 256 else TOTAL_UPDATES)
    dataset = FocusedDataset(list(range(progress['step'], boundary)), resolution, training=True)
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=2,
                        pin_memory=True, persistent_workers=len(dataset)>2)
    generator.train(); discriminator_full.train(); discriminator_half.train()

    for batch in loader:
        sketch = batch['source'].to(DEVICE, non_blocking=True)
        target = batch['target'].to(DEVICE, non_blocking=True)
        next_step = progress['step'] + 1
        with torch.autocast('cuda', dtype=torch.float16):
            fake = generator(sketch)

        gan_active = next_step >= GAN_START
        loss_d = torch.zeros((), device=DEVICE)
        sketch_half = F.interpolate(sketch, scale_factor=.5, mode='bilinear', align_corners=False)
        target_half = F.interpolate(target, scale_factor=.5, mode='bilinear', align_corners=False)
        if gan_active:
            optimizer_d.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                real_full,_ = discriminator_full(torch.cat([sketch,target],1))
                fake_full,_ = discriminator_full(torch.cat([sketch,fake.detach()],1))
                fake_half = F.interpolate(fake.detach(), scale_factor=.5, mode='bilinear', align_corners=False)
                real_half,_ = discriminator_half(torch.cat([sketch_half,target_half],1))
                generated_half,_ = discriminator_half(torch.cat([sketch_half,fake_half],1))
                loss_d = .25*(F.relu(1-real_full).mean()+F.relu(1+fake_full).mean()
                    +F.relu(1-real_half).mean()+F.relu(1+generated_half).mean())
            scaler.scale(loss_d).backward(); scaler.unscale_(optimizer_d)
            nn.utils.clip_grad_norm_(discriminator_parameters, 1.0); scaler.step(optimizer_d)

        for critic in (discriminator_full, discriminator_half): critic.requires_grad_(False)
        optimizer_g.zero_grad(set_to_none=True)
        terms = reconstruction_terms(fake, target)
        feature_matching = torch.zeros((), device=DEVICE)
        adversarial = torch.zeros((), device=DEVICE)
        adversarial_weight = 0.0
        if gan_active:
            with torch.autocast('cuda', dtype=torch.float16):
                score_full, features_full = discriminator_full(torch.cat([sketch,fake],1))
                with torch.no_grad(): _, real_features_full = discriminator_full(torch.cat([sketch,target],1))
                refined_half = F.interpolate(fake, scale_factor=.5, mode='bilinear', align_corners=False)
                score_half, features_half = discriminator_half(torch.cat([sketch_half,refined_half],1))
                with torch.no_grad(): _, real_features_half = discriminator_half(torch.cat([sketch_half,target_half],1))
                feature_matching = sum(F.l1_loss(a,b.detach()) for a,b in zip(
                    features_full+features_half, real_features_full+real_features_half)) / 8
                adversarial = -.5*(score_full.mean()+score_half.mean())
            adversarial_weight = RECIPE['adversarial_max']*min(1.0,(next_step-GAN_START+1)/RECIPE['adversarial_warmup'])
        loss_g = terms['total'] + .50*feature_matching.float() + adversarial_weight*adversarial.float()

        if not torch.isfinite(loss_g) or not torch.isfinite(loss_d):
            save_last(); raise FloatingPointError('Non-finite loss. Last Drive checkpoint is safe.')
        scaler.scale(loss_g).backward(); scaler.unscale_(optimizer_g)
        gradient_norm = nn.utils.clip_grad_norm_(generator.parameters(), 1.0, error_if_nonfinite=False)
        if not torch.isfinite(gradient_norm):
            optimizer_g.zero_grad(set_to_none=True)
            for critic in (discriminator_full, discriminator_half): critic.requires_grad_(True)
            consecutive_nonfinite += 1
            new_scale = max(float(scaler.get_scale())*.5, 1.0); scaler.update(new_scale=new_scale)
            print(f'Skipped unsafe gradient batch; AMP scale reduced to {new_scale:g}.')
            if consecutive_nonfinite >= 3:
                save_last(); raise FloatingPointError('Three unsafe batches. Last Drive checkpoint is safe.')
            continue
        consecutive_nonfinite = 0
        scaler.step(optimizer_g); scaler.update()
        for critic in (discriminator_full, discriminator_half): critic.requires_grad_(True)
        scheduler_g.step()
        if gan_active: scheduler_d.step()

        with torch.no_grad():
            decay = .995 if next_step < 1_000 else .999
            for ema_parameter, parameter in zip(ema_generator.parameters(), generator.parameters()):
                ema_parameter.lerp_(parameter, 1-decay)

        progress['step'] = next_step
        history.append({'type':'train','step':next_step,'resolution':resolution,
            'generator_loss':float(loss_g.detach()), 'paired_loss':float(terms['total'].detach()),
            'pixel_loss':float(terms['pixel'].detach()), 'perceptual_loss':float(terms['perceptual'].detach()),
            'colour_loss':float(terms['colour'].detach()), 'feature_matching':float(feature_matching.detach()),
            'adversarial_loss':float(adversarial.detach()), 'adversarial_weight':adversarial_weight,
            'discriminator_loss':float(loss_d.detach()), 'generator_lr':scheduler_g.get_last_lr()[0]})
        bar.update(1); bar.set_postfix(res=resolution,g=round(float(loss_g.detach()),3),d=round(float(loss_d.detach()),3))

        if next_step % VALIDATE_EVERY == 0 or next_step == TOTAL_UPDATES:
            run_validation()
            if next_step >= 4_000 and progress['bad_validations'] >= EARLY_STOPPING_VALIDATIONS:
                print('Early stopping: validation stopped improving. Best checkpoint is preserved.')
                stop_early = True
        if next_step % SAVE_EVERY == 0 or next_step == TOTAL_UPDATES or stop_early: save_last()
        if next_step == boundary or stop_early: break
    del loader, dataset
    __import__('gc').collect(); torch.cuda.empty_cache()

bar.close(); save_last()
print('Training stopped at:', progress['step'])
print('Validation-selected best step:', progress['best_step'])
print('Automatic quality gate passed:', progress['quality_accepted'])
print('Best candidate:', RUN_ROOT / 'best.pt')

## 6. Teacher-facing learning curves and held-out error analysis

Image-to-image regression has no honest class “accuracy” percentage. Report LPIPS and
MAE (lower is better), plus PSNR and SSIM (higher is better). The plots compare the
unchanged starting U-Net with the validation-selected end-to-end generator on the
untouched test split. Model selection never uses these test scores.

`TEST_LIMIT_PER_DOMAIN=None` evaluates every available test record. Set an integer only
for a clearly labelled preliminary subset when Colab time is limited.

In [ ]:
import csv

BEST_PATH = RUN_ROOT / 'best.pt'
assert BEST_PATH.is_file(), 'Run training first; best.pt does not exist.'
best_state = torch.load(BEST_PATH, map_location='cpu', weights_only=True)
ema_generator.load_state_dict(best_state['generator'], strict=True)
ema_generator.to(DEVICE).eval()

REPORT_ROOT = RUN_ROOT / 'teacher_report'
REPORT_ROOT.mkdir(exist_ok=True)
TEST_LIMIT_PER_DOMAIN = None  # None = full held-out test; integer = labelled subset

# ---- Training and validation curves ----
saved_history = json.loads((RUN_ROOT / 'history.json').read_text())
train_rows = [row for row in saved_history if row.get('type') == 'train']
validation_rows = [row for row in saved_history if row.get('type') == 'validation']

def smoothed(rows, key, window=100):
    x = np.array([row['step'] for row in rows], dtype=float)
    y = np.array([row[key] for row in rows], dtype=float)
    if len(y) < window:
        return x, y
    kernel = np.ones(window) / window
    return x[window-1:], np.convolve(y, kernel, mode='valid')

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for key, label in [('generator_loss','Generator'), ('paired_loss','Paired reconstruction')]:
    x, y = smoothed(train_rows, key)
    axes[0,0].plot(x, y, label=label)
axes[0,0].set(title='Training losses (100-update moving average)', xlabel='Updates', ylabel='Loss')
axes[0,0].legend(); axes[0,0].grid(alpha=.25)

x, y = smoothed(train_rows, 'discriminator_loss')
axes[0,1].plot(x, y, color='tab:red')
axes[0,1].set(title='Discriminator loss (100-update moving average)', xlabel='Updates', ylabel='Hinge loss')
axes[0,1].grid(alpha=.25)

steps = [row['step'] for row in validation_rows]
axes[1,0].plot(steps, [row['fs2k_artist']['lpips'] for row in validation_rows], marker='o', label='FS2K portraits')
axes[1,0].plot(steps, [row['coco_synthetic']['lpips'] for row in validation_rows], marker='o', label='COCO scenes')
axes[1,0].set(title='Validation LPIPS — lower is better', xlabel='Updates', ylabel='LPIPS')
axes[1,0].legend(); axes[1,0].grid(alpha=.25)

axes[1,1].plot(steps, [row['balanced_score'] for row in validation_rows], marker='o', color='tab:purple')
axes[1,1].axvline(best_state['step'], color='tab:green', linestyle='--', label=f"Selected step {best_state['step']}")
axes[1,1].set(title='Domain-balanced validation score', xlabel='Updates', ylabel='Score (lower is better)')
axes[1,1].legend(); axes[1,1].grid(alpha=.25)
fig.tight_layout(); fig.savefig(REPORT_ROOT / 'training_validation_curves.png', dpi=160)
plt.show(); plt.close(fig)

# ---- Full-reference metrics on untouched test records ----
def uniform_ssim(prediction_01, target_01, window=11):
    mu_x = F.avg_pool2d(prediction_01, window, 1)
    mu_y = F.avg_pool2d(target_01, window, 1)
    sigma_x = F.avg_pool2d(prediction_01.square(), window, 1) - mu_x.square()
    sigma_y = F.avg_pool2d(target_01.square(), window, 1) - mu_y.square()
    sigma_xy = F.avg_pool2d(prediction_01 * target_01, window, 1) - mu_x * mu_y
    c1, c2 = .01**2, .03**2
    value = ((2*mu_x*mu_y+c1) * (2*sigma_xy+c2)) / ((mu_x.square()+mu_y.square()+c1) * (sigma_x+sigma_y+c2))
    return value.mean()

def metric_row(prediction, target, model_name, record, box):
    left, top, right, bottom = box
    prediction = prediction[:, :, top:bottom, left:right].float().clamp(-1,1)
    target = target[:, :, top:bottom, left:right].float().clamp(-1,1)
    if min(prediction.shape[-2:]) < 64:
        raise ValueError('content crop is too narrow for reliable LPIPS')
    with torch.autocast('cuda', dtype=torch.float16):
        perceptual = float(PERCEPTUAL(prediction, target).mean())
    prediction_01, target_01 = (prediction+1)/2, (target+1)/2
    mae = float(F.l1_loss(prediction_01, target_01))
    mse = float(F.mse_loss(prediction_01, target_01))
    psnr = float(10 * math.log10(1 / max(mse, 1e-12)))
    ssim = float(uniform_ssim(prediction_01, target_01))
    return {'id':record['id'], 'domain':record['domain'], 'model':model_name,
            'lpips':perceptual, 'mae':mae, 'psnr_db':psnr, 'ssim':ssim}

test_records = []
for domain_rows in (FS2K_SPLITS['test'], COCO_SPLITS['test']):
    rows = domain_rows if TEST_LIMIT_PER_DOMAIN is None else domain_rows[:TEST_LIMIT_PER_DOMAIN]
    test_records.extend(rows)

test_rows, excluded_rows = [], []
for record in tqdm(test_records, desc='Held-out test evaluation'):
    try:
        photo, sketch, box = load_pair(record, 512)
        source = tensor_image(sketch)[None].to(DEVICE)
        target = tensor_image(photo)[None].to(DEVICE)
        with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
            baseline, refined = predict_tensor(source, ema_generator)
        test_rows.append(metric_row(baseline, target, 'Starting U-Net', record, box))
        test_rows.append(metric_row(refined, target, 'Fine-tuned generator', record, box))
    except Exception as error:
        excluded_rows.append({'id':record['id'], 'domain':record['domain'], 'reason':str(error)})

assert test_rows, 'No test records were evaluated.'
scope = 'full_test' if TEST_LIMIT_PER_DOMAIN is None else f'subset_{TEST_LIMIT_PER_DOMAIN}_per_domain'
summary = []
for model_name in ('Starting U-Net', 'Fine-tuned generator'):
    for domain in ('fs2k_artist', 'coco_synthetic'):
        rows = [row for row in test_rows if row['model']==model_name and row['domain']==domain]
        assert rows, f'No evaluated rows for {model_name}, {domain}'
        summary.append({
            'scope':scope, 'model':model_name, 'domain':domain, 'n':len(rows),
            **{metric:float(np.mean([row[metric] for row in rows]))
               for metric in ('lpips','mae','psnr_db','ssim')},
        })

with (REPORT_ROOT / 'test_metrics_per_image.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(test_rows[0]))
    writer.writeheader(); writer.writerows(test_rows)
with (REPORT_ROOT / 'test_summary.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary[0]))
    writer.writeheader(); writer.writerows(summary)
atomic_json(REPORT_ROOT / 'test_summary.json', {'scope':scope, 'summary':summary, 'excluded':excluded_rows})

print('Held-out evaluation scope:', scope)
for row in summary:
    print(row)
if excluded_rows:
    print('Excluded records (with reasons):', len(excluded_rows))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
metric_specs = [('lpips','LPIPS ↓'), ('mae','MAE ↓'), ('psnr_db','PSNR dB ↑'), ('ssim','SSIM ↑')]
models = ['Starting U-Net', 'Fine-tuned generator']
domains = ['fs2k_artist', 'coco_synthetic']
for axis, (metric, title) in zip(axes.flat, metric_specs):
    x = np.arange(len(domains)); width = .36
    for index, model_name in enumerate(models):
        values = [next(row[metric] for row in summary if row['model']==model_name and row['domain']==domain) for domain in domains]
        axis.bar(x + (index-.5)*width, values, width, label=model_name)
    axis.set_xticks(x, ['FS2K portraits','COCO scenes']); axis.set_title(title); axis.grid(axis='y', alpha=.2)
axes[0,0].legend()
fig.suptitle('Held-out test comparison — ' + scope)
fig.tight_layout(); fig.savefig(REPORT_ROOT / 'test_metric_comparison.png', dpi=160)
plt.show(); plt.close(fig)

# ---- Worst-case error examples for honest failure analysis ----
refined_rows = [row for row in test_rows if row['model']=='Fine-tuned generator']
worst = sorted(refined_rows, key=lambda row:row['lpips'], reverse=True)[:6]
record_lookup = {record['id']:record for record in test_records}
fig, axes = plt.subplots(len(worst), 4, figsize=(14, 3.6*len(worst)))
for row_index, metric in enumerate(worst):
    record = record_lookup[metric['id']]
    photo, sketch, box = load_pair(record, 512)
    source = tensor_image(sketch)[None].to(DEVICE)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        baseline, refined = predict_tensor(source, ema_generator)
    images = [sketch.crop(box), tensor_to_image(baseline).crop(box),
              tensor_to_image(refined).crop(box), photo.crop(box)]
    titles = ['Input sketch','Starting U-Net',f"Fine-tuned — LPIPS {metric['lpips']:.3f}",'Real target']
    for axis, image, title in zip(axes[row_index], images, titles):
        axis.imshow(image); axis.set_title(title); axis.axis('off')
fig.tight_layout(); fig.savefig(REPORT_ROOT / 'worst_test_failures.png', dpi=150)
plt.show(); plt.close(fig)

print('Teacher report folder:', REPORT_ROOT)
print('- training_validation_curves.png')
print('- test_metric_comparison.png')
print('- test_summary.csv / test_summary.json')
print('- test_metrics_per_image.csv')
print('- worst_test_failures.png')

## 7. Final visual gate and export

Run only after reviewing Section 6. The export records whether the automatic gate passed,
but **you must also reject it manually** if faces, colours, or general scenes still look
unacceptable. Training completion is not proof of quality.

In [ ]:
BEST_PATH = RUN_ROOT / 'best.pt'
assert BEST_PATH.is_file(), 'Run the training cell first.'
best_state = torch.load(BEST_PATH, map_location='cpu', weights_only=True)
ema_generator.load_state_dict(best_state['generator'], strict=True)
ema_generator.to(DEVICE).eval()
render_gate(ema_generator, best_state['step'], 'FINAL_VISUAL_GATE.png')

manifest = {
    'format':'sketch2photo-end-to-end-v3', 'direction':'sketch_to_photo',
    'quality_status':('Automatic numeric gate passed; human visual approval still required.'
                      if best_state.get('quality_accepted') else
                      'REJECTED by automatic gate; retain only as experiment evidence.'),
    'best_step':best_state['step'], 'validation':best_state['validation'],
    'quality_accepted':bool(best_state.get('quality_accepted',False)),
    'checkpoint':'best.pt', 'starting_source':str(STARTING_CHECKPOINT),
    'photo_to_pencil_status':'Untouched; continue using the previously accepted checkpoint.',
}
atomic_json(RUN_ROOT / 'manifest.json', manifest)

export_root = RUN_ROOT / 'export'; export_root.mkdir(exist_ok=True)
for source in [BEST_PATH, RUN_ROOT/'manifest.json', RUN_ROOT/'recipe.json', RUN_ROOT/'FINAL_VISUAL_GATE.png']:
    shutil.copy2(source, export_root/source.name)
for report_path in REPORT_ROOT.iterdir():
    if report_path.is_file(): shutil.copy2(report_path, export_root/report_path.name)
archive = shutil.make_archive(str(RUN_ROOT/'sketch_to_photo_v3_export'), 'zip', export_root)
print('Quality accepted:', manifest['quality_accepted'])
print('Review:', RUN_ROOT/'FINAL_VISUAL_GATE.png')
print('Drive export:', archive)
print('Do not replace the application model unless the visual gate is acceptable.')

## Resume and decision checklist

- Upload this notebook to Colab and run cells in order from the top.
- If Colab reaches its limit, reconnect with a GPU and rerun from the top; Section 5 resumes.
- Inspect gates at steps 500, 1000, 1500, 2000, and later.
- Continue past 2000 only if portraits are less pink/blurry **and** scenes remain stable.
- Run Section 6 for the teacher metrics even if you stop/reject training.
- Section 7 packages evidence but does not automatically change the local application.
- The accepted photo → pencil model and all v1/v2 checkpoints remain untouched.